In [ ]:
from dotenv import load_dotenv
import os
import time  # <--- agrega esta línea
import json
import requests

# Ruta .env en la raíz del repo
env_path = os.path.abspath(os.path.join(os.getcwd(), "../../..", ".env"))
load_dotenv(dotenv_path=env_path)

# Obtener clave de Grok desde .env
GROK_API_KEY = os.getenv("GROK_API_KEY")

if GROK_API_KEY:
    print("Clave de Grok cargada correctamente.")
else:
    raise ValueError("❌ No se encontró la clave de Grok. Verifica la ruta del .env.")

Clave de Grok cargada correctamente.


In [ ]:
def call_grok_api(prompt, text):
    """
    Envía un texto y un prompt al modelo Grok-4 y devuelve el resumen generado.
    """
    url = "https://api.x.ai/v1/chat/completions"

    headers = {
        "Authorization": f"Bearer {GROK_API_KEY}",
        "Content-Type": "application/json"
    }

    payload = {
        "model": "grok-4",
        "messages": [
            {"role": "system", "content": "Eres un asistente experto en simplificar lenguaje biomédico."},
            {"role": "user", "content": f"{prompt}\n\nTexto:\n{text}"}
        ]
    }

    try:
        start_time = time.time()
        response = requests.post(url, headers=headers, json=payload)
        elapsed = time.time() - start_time

        # Mostrar más información si algo falla
        print(f"Código de estado: {response.status_code}")
        try:
            print("Respuesta (primeros 200 caracteres):", response.text[:200])
        except Exception as e:
            print("No se pudo imprimir el texto:", e)

        if response.status_code == 200:
            data = response.json()
            # Muestra las claves del JSON
            print("Claves en respuesta:", data.keys())
            output = data["choices"][0]["message"]["content"]
            return output.strip(), elapsed
        else:
            print(f"Error HTTP {response.status_code}: {response.text}")
            return None, elapsed
    except Exception as e:
        print("Error en la solicitud:", e)
        return None, None

In [ ]:
import pandas as pd
#traemos el archivo test.csv
ruta_dataset = "../data-sources/pre-processed/data_finetuning_test.csv"
df = pd.read_csv(ruta_dataset, encoding="utf-8", on_bad_lines="skip")
display(df.head(2))

,name,article,summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...


In [ ]:
# Prompt a utilizarse
prompt = """You are a helpful medical/health writer.
Summarize the following scientific text into a clear summary intended for a general audience.
Do NOT use headings, titles, bullet points, or numbered lists. Do not invent data or references.
If you must use a technical term, briefly define it."""

# Tomar las dos primeras filas para prueba inicial
df_prueba = df.head(2).copy()

# Lista donde guardaremos los resúmenes generados de la prueba
gen_summaries = []

for i, fila in df_prueba.iterrows():
    print(f"\n Procesando {fila['name']} ({i+1}/{len(df_prueba)})...\n")
    resumen, tiempo = call_grok_api(prompt, fila['article'])
    gen_summaries.append(resumen if resumen else "")
    print(f"Tiempo de respuesta: {tiempo:.2f} s\n")

# Añadir columna con los resúmenes generados
df_prueba["gen_summary"] = gen_summaries

# Guardar en CSV la prueba
ruta_salida = "./resultados_grok.csv"
df_prueba.to_csv(ruta_salida, index=False, encoding="utf-8")

print(f"Resultados guardados en: {os.path.abspath(ruta_salida)}")
display(df_prueba)


📄 Procesando 10.1002-14651858.CD009781.pub2 (1/2)...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"f9244f61-93c9-1110-ff4e-d23bcb06cf72_us-east-1","object":"chat.completion","created":1761481028,"model":"grok-4-0709","choices":[{"index":0,"message":{"role":"assistant","content":"Traumatic co
Claves en respuesta: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'system_fingerprint'])
⏱️ Tiempo de respuesta: 22.54 s


📄 Procesando 10.1002-14651858.CD010694.pub2 (2/2)...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"444c7a13-19a3-5497-1010-3ba96d6d5136_us-east-1","object":"chat.completion","created":1761481051,"model":"grok-4-0709","choices":[{"index":0,"message":{"role":"assistant","content":"Venous leg u
Claves en respuesta: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'system_fingerprint'])
⏱️ Tiempo de respuesta: 20.17 s

✅ Resultados guardados en: c:\Users\braya\OneDrive\Documentos\GitHub\Proyecto-PLN-

,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,"Traumatic corneal abrasions, which are scratch..."
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,"Venous leg ulcers are common, long-lasting wou..."


In [ ]:
#Aca vamos a usar todo el dataset
df_groktest = df.copy()

# Lista donde guardaremos los resúmenes generados
gen_summaries = []

# Contador de tiempo total
inicio_total = time.time()

for i, fila in df_groktest.iterrows():
    print(f"\n Procesando {fila['name']} ({i+1}/{len(df_groktest)})...\n")
    resumen, tiempo = call_grok_api(prompt, fila['article'])
    gen_summaries.append(resumen if resumen else "")
    print(f" Tiempo de respuesta: {tiempo:.2f} s\n")

# Añadir columna con los resúmenes generados
df_groktest["gen_summary"] = gen_summaries

# Guardar resultados
ruta_salida = "./results_grok.csv"
df_groktest.to_csv(ruta_salida, index=False, encoding="utf-8")

duracion_total = time.time() - inicio_total
print(f"Resultados guardados en: {os.path.abspath(ruta_salida)}")
print(f"Tiempo total: {duracion_total/60:.2f} minutos ({duracion_total/len(df_groktest):.2f} s por fila en promedio)")


 Procesando 10.1002-14651858.CD009781.pub2 (1/380)...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"a6ea987c-8c14-b180-3908-127181ece4e0_us-east-1","object":"chat.completion","created":1761481071,"model":"grok-4-0709","choices":[{"index":0,"message":{"role":"assistant","content":"Traumatic co
Claves en respuesta: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'system_fingerprint'])
 Tiempo de respuesta: 32.40 s


 Procesando 10.1002-14651858.CD010694.pub2 (2/380)...

Código de estado: 200
Respuesta (primeros 200 caracteres): {"id":"30c56dd2-a9d5-2072-c8d5-b1865248e654_us-east-1","object":"chat.completion","created":1761481103,"model":"grok-4-0709","choices":[{"index":0,"message":{"role":"assistant","content":"Venous leg u
Claves en respuesta: dict_keys(['id', 'object', 'created', 'model', 'choices', 'usage', 'system_fingerprint'])
 Tiempo de respuesta: 23.45 s


 Procesando 10.1002-14651858.CD009416.pub2 (3/380)...

Código de estado: 200
Respue

KeyboardInterrupt: 

In [10]:
# Ruta del archivo csv guardado para verificar su contenido.
ruta_csv = "./results_grok.csv"
df_check = pd.read_csv(ruta_csv)


df_check.info()

df_check.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 380 entries, 0 to 379
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   name         380 non-null    object
 1   article      380 non-null    object
 2   summary      380 non-null    object
 3   gen_summary  380 non-null    object
dtypes: object(4)
memory usage: 12.0+ KB


,name,article,summary,gen_summary
0,10.1002-14651858.CD009781.pub2,Background\r\nTraumatic corneal abrasions are ...,Topical non‐steroidal anti‐inflammatory drugs ...,"Traumatic corneal abrasions, which are scratch..."
1,10.1002-14651858.CD010694.pub2,"Background\r\nVenous leg ulcers are common, ch...",Sulodexide for venous leg ulcers\r\nReview que...,"Venous leg ulcers are common, long-lasting wou..."
2,10.1002-14651858.CD009416.pub2,Background\r\nThere is currently no strong con...,Which treatments are effective for the treatme...,"Complex regional pain syndrome, or CRPS, is a ..."
3,10.1002-14651858.CD004104.pub4,Background\r\nNon‐invasive ventilation (NIV) w...,Non‐invasive ventilation for people with respi...,"Non-invasive ventilation, often delivered thro..."
4,10.1002-14651858.CD012689.pub2,Background\r\nSpace spraying is the dispersal ...,Insecticide space spraying for preventing mala...,Space spraying involves releasing a fog of ins...
